[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_2_Lagrange_Method_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


# 🧭 اللاب 2: المعادلات شبه الخطية وطريقة لاجرانج لخطوط الانسياب
### *غرفة التحكم: محاكي تتبع مسار عجلات المركبة*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**موضوع المنهج:** الباب الأول، القسم 1.3.1 (صفحات 12–16)

---
### 🧭 حجر رشيد (The Rosetta Stone): ترجمة الرياضيات إلى لغة البشر وكود بايثون
| 1. معادلات السبورة في المحاضرة | 2. المعنى الفيزيائي البسيط | 3. كود بايثون المقابل |
| :--- | :--- | :--- |
| $P p + Q q = R$ | "معادلة نقل تدفق الرياح أو السرعات في اتجاهين." | `u*dphi_dx + v*dphi_dy = 0` |
| $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$ | "معادلات لاجرانج المساعدة: مسار حركة الجسيم خطوة بخطوة." | `dx / u == dy / v` |
| $\Phi(u, v) = 0$ | "العلاقة التي تثبت أن مسار المركبة يتبع خط انسياب ثابت." | `y = k * x**2` |

---
## 1. المعنى البسيط: "الملاحة مع تيار النهر"
تخيل أنك تركب قارباً صغيراً في نهر متدفق. بدلاً من مقاومة التيار، تترك القارب ينساب تلقائياً مع خطوط التيار (Streamlines).
* **طريقة لاجرانج (Lagrange's Method):** هي الأداة الرياضية التي تحول المعادلة التفاضلية الجزئية ثلاثية الأبعاد إلى معادلات تفاضلية عادية (ODEs) تصف مسار حركة العجلات أو جزيئات الغبار على التضاريس خطوة بخطوة!

---
## 2. ما فائدة هذه الطريقة ولماذا نقوم بها؟ (The "WHY")
* **لماذا نقوم بها؟** المركبة بعد حساب مجال الجهد $\phi(x, y)$ تحتاج إلى قانون قيادة محدد: كيف تدير عجلاتها في كل ثانية؟
* **ماذا تفيد مركبة المريخ؟** عندما تحسب متجه الانحدار $\mathbf{v} = (u, v) = -\nabla\phi$، فإن معادلة حركة خط الانسياب هي $\frac{dx}{u} = \frac{dy}{v}$. حل هذه المعادلة بلاجرانج هو **بالضبط ما نكتبه في كود دالة `descend()` لتوجيه المركبة!**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab2

%matplotlib inline
print("تم تهيئة بيئة اللاب 2 بنجاح! جاهز لتتبع خطوط الانسياب.")


---
## 3. الوصفة الرياضية ومثال المذكرة المحلول (Worked Example)

لحل أي معادلة من الصورة $P p + Q q = R$:
* **الخطوة 1:** كتابة معادلات لاجرانج المساعدة: $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$.
* **الخطوة 2:** أخذ زوجين منفصلين وتكاملهما لإيجاد ثابتين مستقلين $u(x, y, z) = c_1$ و $v(x, y, z) = c_2$.
* **الخطوة 3:** كتابة الحل العام: $\Phi(c_1, c_2) = 0$.

### مثال محلول من مذكرات المحاضرة (صفحة 13):
$$2 p - 3 q = 2x$$
* $P = 2, Q = -3, R = 2x$.
* $\frac{dx}{2} = \frac{dy}{-3} \implies -3 dx = 2 dy \implies 3x + 2y = c_1$.
* $\frac{dx}{2} = \frac{dz}{2x} \implies 2x dx = 2 dz \implies x^2 - 2z = c_2$.
* **الحل العام:** $\Phi(3x + 2y, x^2 - 2z) = 0$.


---
## 4. 📝 تحدي الورقة والقلم 1 (مسألة من مذكرات المنهج)

> 💡 *(إذا واجهتك صعوبة، راجع `INSTRUCTOR_ANSWER_KEY.md` القسم 2.1).*

**المسألة (معادلة أويلر المتجانسة):**
$$x p + y q = z$$
1. اكتب معادلات لاجرانج المساعدة: $\frac{dx}{x} = \frac{dy}{y} = \frac{dz}{z}$.
2. خذ الكسرين 1 و 2 ($\frac{dx}{x} = \frac{dy}{y}$) وكاملهما لإيجاد $u(x, y) = c_1$.
3. خذ الكسرين 2 و 3 ($\frac{dy}{y} = \frac{dz}{z}$) وكاملهما لإيجاد $v(y, z) = c_2$.

اكتب صيغتي $u$ و $v$ كنصوص أدناه (مثال: `"x/y"` و `"y/z"`):


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_u = "x/y"
student_v = "y/z"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab2.check_paper_challenge_1(student_u, student_v)


---
## 5. 🚀 تحدي جسر مركبة المريخ (مسارات عجلات المركبة في الرمال)

افترض أن المركبة تسير على حقل جهد $\phi(x, y) = x^2 + 2y^2$.  
متجه السرعة مع تيار الانحدار هو $\mathbf{v} = (u, v) = -\nabla\phi = (-2x, -4y)$.  
معادلة نقل مسار الانسياب هي:
$$-2x \frac{\partial \phi}{\partial x} - 4y \frac{\partial \phi}{\partial y} = 0$$

**على ورقتك:**
1. اكتب معادلة لاجرانج: $\frac{dx}{-2x} = \frac{dy}{-4y}$.
2. بسط المقامات بالقسمة على $-2$: $\frac{dx}{x} = \frac{dy}{2y}$.
3. كامل الطرفين: $\ln |x| = \frac{1}{2}\ln |y| + C \implies y = k x^2$.

ما هو المنحنى الذي تتبعه العجلات؟ اكتب المعادلة أدناه:


In [ ]:
# خلية تقييم: مسار عجلات المركبة
### START CODE HERE ###
rover_streamline = "y = k*x^2"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab2.check_paper_challenge_2(rover_streamline)


---
## 6. 💻 التطبيق البرمجي ببايثون: متتبع خطوط الانسياب العددي (RK4)

سنكتب دالة تحاكي حركة عجلات المركبة بتكامل متجه السرعة عددياً بخطوات صغيرة $\Delta t$:


In [ ]:
# دالة تقييم برمجية: trace_streamline_rk4
def trace_streamline_rk4(velocity_fn, start=(1.0, 2.0), dt=0.01, steps=50):
    # تتبع خط انسياب نظام شبه خطي عددياً
    trajectory = [start]
    curr = np.array(start, dtype=float)
    
    ### START CODE HERE ###
    for _ in range(steps):
        vx, vy = velocity_fn(curr[0], curr[1])
        v = np.array([vx, vy], dtype=float)
        curr = curr + dt * v
        trajectory.append(tuple(curr))
    ### END CODE HERE ###
    
    return np.array(trajectory)

# تشغيل الاختبار الآلي:
test_lab2.test_rk4_streamline_tracer(trace_streamline_rk4)


---
## 7. 🎨 الرؤية البصرية للفيزياء: خطوط الانسياب الحقيقية
شغّل الخلية التالية لرؤية خطوط الانسياب المحسوبة بطريقة لاجرانج وهي تطابق حركة المركبة:


In [ ]:
def sample_velocity(x, y):
    return -2.0 * x, -4.0 * y

traj = trace_streamline_rk4(sample_velocity, start=(1.0, 2.0), dt=0.01, steps=100)

x_vals = np.linspace(-2, 2, 20)
y_vals = np.linspace(-1, 3, 20)
X, Y = np.meshgrid(x_vals, y_vals)
U, V = sample_velocity(X, Y)

plt.figure(figsize=(8, 6))
plt.streamplot(X, Y, U, V, color='gray', density=0.8)
plt.plot(traj[:, 0], traj[:, 1], 'r-', linewidth=3, label="Lagrange Characteristic Trajectory")
plt.plot(traj[0, 0], traj[0, 1], 'go', markersize=8, label="Start (1, 2)")
plt.title("Rover Characteristic Streamlines: Lagrange Method in Action", fontsize=12)
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(True)
plt.show()

print("تهانينا! لقد أتممت اللاب 2 وأتقنت طريقة لاجرانج لخطوط الانسياب.")
